<a href="https://colab.research.google.com/github/misrori/ai/blob/2025/video_feliratozo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai==0.28
!pip install openai pydub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 2.2 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.61.1
    Uninstalling openai-1.61.1:
      Successfully uninstalled openai-1.61.1


In [ ]:
import os
import time
import json
import subprocess
from datetime import timedelta
import openai
from pydub import AudioSegment
import math

# OpenAI API kulcs beállítása
openai.api_key = "sk-proj-"

# Maximum fájlméret byte-ban (25MB - kicsit kisebbre állítva biztonság kedvéért)
MAX_FILE_SIZE = 25 * 1024 * 1024

def format_time(seconds):
    """Másodperceket SRT időformátumba konvertál (HH:MM:SS,mmm)"""
    td = timedelta(seconds=seconds)
    hours, remainder = divmod(td.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    milliseconds = int(td.microseconds / 1000)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d},{milliseconds:03d}"

def extract_audio_from_video(video_file_path, audio_file_path="temp_audio.mp3"):
    """Hang kinyerése videófájlból"""
    print(f"Hang kinyerése a videóból: {video_file_path}")

    try:
        # FFmpeg használata a hang kinyeréséhez
        subprocess.run([
            "ffmpeg", "-y", "-i", video_file_path, "-q:a", "0", "-map", "a", audio_file_path
        ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

        print(f"Hang sikeresen kinyerve: {audio_file_path}")
        return audio_file_path
    except subprocess.CalledProcessError as e:
        print(f"Hiba történt a hang kinyerése során: {str(e)}")
        return None

def split_audio(audio_file_path, chunk_length_ms=600000):
    """Hangfájl felosztása kisebb darabokra

    Args:
        audio_file_path: A hangfájl útvonala
        chunk_length_ms: Egy részlet hossza millimásodpercben (alapértelmezett: 10 perc)

    Returns:
        temp_files: Ideiglenes fájlok listája
        total_duration_ms: A teljes hangfájl hossza ezredmásodpercben
    """
    print(f"Hangfájl darabolása: {audio_file_path}")

    # Hangfájl betöltése
    audio = AudioSegment.from_file(audio_file_path)
    total_duration_ms = len(audio)

    # Hány darabra kell osztani
    file_size = os.path.getsize(audio_file_path)
    num_chunks = math.ceil(file_size / MAX_FILE_SIZE)
    chunk_length_ms = min(chunk_length_ms, math.ceil(total_duration_ms / num_chunks))

    # Darabolás és ideiglenes fájlok mentése
    temp_files = []
    for i, chunk_start in enumerate(range(0, len(audio), chunk_length_ms)):
        chunk_end = min(chunk_start + chunk_length_ms, len(audio))
        chunk = audio[chunk_start:chunk_end]

        # Ideiglenes fájl készítése
        chunk_file = f"temp_chunk_{i}.mp3"
        chunk.export(chunk_file, format="mp3")
        temp_files.append((chunk_file, chunk_start / 1000))  # Kezdő idő másodpercben

        print(f"Részlet {i+1} mentve: {chunk_file}")

    return temp_files, total_duration_ms / 1000

def transcribe_audio_chunk(audio_chunk_path):
    """Egy hangfájl részlet átírása"""
    try:
        with open(audio_chunk_path, "rb") as audio_file:
            transcription = openai.Audio.transcribe(
                "whisper-1",
                audio_file,
                response_format="verbose_json"
            )
        return transcription
    except Exception as e:
        print(f"Hiba történt a részlet átírása során: {str(e)}")
        return None

def create_srt_from_audio(audio_file_path, output_srt_path):
    """Hangfájlból SRT feliratot készít, szükség esetén darabolva"""
    print(f"Hangfájl feldolgozása: {audio_file_path}")

    # A fájlméret ellenőrzése
    file_size = os.path.getsize(audio_file_path)

    if file_size <= MAX_FILE_SIZE:
        # Ha a fájl mérete megfelelő, egyszerűen átírjuk
        try:
            with open(audio_file_path, "rb") as audio_file:
                transcription = openai.Audio.transcribe(
                    "whisper-1",
                    audio_file,
                    response_format="verbose_json"
                )

            # SRT fájl készítése
            create_srt_from_transcription(transcription, output_srt_path)
        except Exception as e:
            print(f"Hiba történt az átírás során: {str(e)}")
            return False
    else:
        # Ha túl nagy a fájl, feldaraboljuk és részenként írjuk át
        temp_chunks, total_duration = split_audio(audio_file_path)

        # SRT fájl előkészítése
        with open(output_srt_path, "w", encoding="utf-8") as srt_file:
            subtitle_index = 1

            for chunk_file, start_offset in temp_chunks:
                transcription = transcribe_audio_chunk(chunk_file)

                if transcription and hasattr(transcription, "segments"):
                    # Részletek hozzáadása
                    for segment in transcription.segments:
                        start_time = segment.start + start_offset
                        end_time = segment.end + start_offset
                        text = segment.text.strip()

                        # SRT bejegyzés írása
                        srt_file.write(f"{subtitle_index}\n")
                        srt_file.write(f"{format_time(start_time)} --> {format_time(end_time)}\n")
                        srt_file.write(f"{text}\n\n")

                        subtitle_index += 1

                # Ideiglenes fájl törlése
                os.remove(chunk_file)

    print(f"SRT felirat sikeresen elkészült: {output_srt_path}")
    return True

def create_srt_from_transcription(transcription, output_path):
    """Átírásból SRT fájlt készít"""
    with open(output_path, "w", encoding="utf-8") as srt_file:
        if hasattr(transcription, "segments"):
            for i, segment in enumerate(transcription.segments):
                start_time = segment.start
                end_time = segment.end
                text = segment.text.strip()

                # SRT bejegyzés írása
                srt_file.write(f"{i+1}\n")
                srt_file.write(f"{format_time(start_time)} --> {format_time(end_time)}\n")
                srt_file.write(f"{text}\n\n")
        else:
            # Ha nincs időzítés, egyszerű SRT készítése
            srt_file.write("1\n")
            srt_file.write("00:00:00,000 --> 00:05:00,000\n")
            srt_file.write(transcription.text + "\n\n")

    print(f"SRT felirat sikeresen elkészült: {output_path}")

def add_subtitles_to_video(video_file_path, srt_file_path, output_mkv_path):
    """Feliratok hozzáadása a videóhoz és MKV formátumban mentése"""
    print(f"Feliratok hozzáadása a videóhoz: {video_file_path}")

    try:
        # FFmpeg használata a feliratok hozzáadásához
        subprocess.run([
            "ffmpeg", "-y", "-i", video_file_path, "-i", srt_file_path,
            "-map", "0", "-map", "1", "-c", "copy", "-c:s", "srt",
            output_mkv_path
        ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

        print(f"Videó feliratozva és mentve: {output_mkv_path}")
        return True
    except subprocess.CalledProcessError as e:
        print(f"Hiba történt a feliratok hozzáadása során: {str(e)}")
        return False

def process_video_file(video_file_path):
    """Teljes videófájl feldolgozás: hang kinyerése, átírás, feliratozás"""
    # Kimeneti fájlnév alapértelmezett értéke

    base_name = os.path.splitext(os.path.basename(video_file_path))[0]
    output_mkv_path = f"{base_name}_feliratozott.mkv"

    # Ideiglenes fájlnevek
    temp_audio_path = "temp_audio.mp3"
    temp_srt_path = "temp_felirat.srt"

    # 1. Hang kinyerése a videóból
    audio_path = extract_audio_from_video(video_file_path, temp_audio_path)
    if not audio_path:
        print("A folyamat megszakítva a hang kinyerése során történt hiba miatt.")
        return False

    # 2. SRT felirat készítése a hangból
    srt_success = create_srt_from_audio(audio_path, temp_srt_path)
    if not srt_success:
        print("A folyamat megszakítva az átírás során történt hiba miatt.")
        # Ideiglenes hangfájl törlése
        if os.path.exists(temp_audio_path):
            os.remove(temp_audio_path)
        return False

    # 3. Feliratok hozzáadása a videóhoz és MKV formátumban mentése
    subtitle_success = add_subtitles_to_video(video_file_path, temp_srt_path, output_mkv_path)

    # 4. Ideiglenes fájlok törlése
    if os.path.exists(temp_audio_path):
        os.remove(temp_audio_path)

    # Az SRT fájlt megtartjuk, mert hasznos lehet külön is

    if subtitle_success:
        print(f"A videó feldolgozása sikeres! Feliratozva és mentve: {output_mkv_path}")
        return True
    else:
        print("A folyamat megszakítva a feliratok hozzáadása során történt hiba miatt.")
        return False

# Példa használat
if __name__ == "__main__":
    video_file_path = 'mp.mp4'  # pl. "video.mp4" vagy "video.mkv"


    process_video_file(video_file_path)

Hang kinyerése a videóból: mp.mp4
Hang sikeresen kinyerve: temp_audio.mp3
Hangfájl feldolgozása: temp_audio.mp3
SRT felirat sikeresen elkészült: temp_felirat.srt
SRT felirat sikeresen elkészült: temp_felirat.srt
Feliratok hozzáadása a videóhoz: mp.mp4
Videó feliratozva és mentve: mp_feliratozott.mkv
A videó feldolgozása sikeres! Feliratozva és mentve: mp_feliratozott.mkv
